#### Imports

In [318]:
from student.agent import *
from student.agent.memory_rag import MemoryRAG, MemoryNodeRAG
import pandas as pd
import json
from student.agent.agent_baselines import BaselineAgent
import random

### Config

In [39]:
training_run_id = "001"

In [ ]:
RAG_MEMORY_PATHS = {"fictsheets": "memory/fictsheets_fqa_rag.parquet", "fictions": "memory/fictions_fqa_rag.parquet"}
TRAINING_STUDENT_MEMORY_PATH = f"checkpoints/training_{training_run_id}"

AGENT_CONFIG = {
    "expensive": False,
    "provider": "anthropic",
    "cache": False
}

# Prepare Memories: FictionalQA

### Data

In [ ]:
data = pd.read_parquet("hf://datasets/tomg-group-umd/fictionalqa/fict_qa/train-00000-of-00001.parquet")
fictsheets = pd.read_parquet("hf://datasets/tomg-group-umd/fictionalqa/fictsheets/train-00000-of-00001.parquet")
fqa_mc = pd.read_parquet('https://huggingface.co/api/datasets/tomg-group-umd/fictionalqa_training_splits/parquet/fict_qa_obqa_blind_inf_ex_dedup_ds_Llama-3-2-3B-Instruct_scored_rowlimNone_altlimNone_topk4_seed1234_slim/train/0.parquet')
fictions = pd.read_parquet("hf://datasets/tomg-group-umd/fictionalqa/fictions/train-00000-of-00001.parquet")

def data_from_question_id(question_id, data) -> pd.core.frame.DataFrame:
    entry = data[data['question_id'] == question_id]
    return entry[["event_id", "fiction_id", "question_id", "question_num", "fict", "question", "natural_answer"]]

# data_from_question_id("event_000_style_blog_num_000_question_003", data)

def load_fictsheets(event_id, fictsheets) -> str:
    # Returns the fictsheet as str
    fictsheets = fictsheets[["event_id", "fictsheet"]]
    return fictsheets[fictsheets['event_id'] == event_id]["fictsheet"].values[0]

# load_fictsheets("event_000", fictsheets)

def question_id_to_style(question_id) -> tuple[str, str]:
    # question_id: event_xxx_style_blog_num_xxx_question_xxx
    parts = question_id.split("_")
    style = parts[3]
    style_num = parts[5]
    return style, style_num

# question_id_to_style("event_000_style_blog_num_000_question_003")

def fqa_row_to_data(row, data, fictsheets) -> dict:
    question_id = row["question_id"]
    data_entry = data_from_question_id(question_id, data).to_dict()
    question_num = list(data_entry["question_num"].keys())[0]

    return {
        "event_id": row["event_id"],
        "fictsheet": load_fictsheets(row["event_id"], fictsheets),
        "question": data_entry["question"][question_num],
        "natural_answer": data_entry["natural_answer"][question_num],
        "question_id": data_entry["question_id"][question_num],
        "question_num": data_entry["question_num"][question_num],
        "fict": data_entry["fict"][question_num],
    }
# fqa_row_to_data(row, data, fictsheets)

In [ ]:
n_event_ids = 100

In [ ]:
styles = [
    '_style_blog_num_000',
    '_style_blog_num_001',
    '_style_corporate_num_000',
    '_style_corporate_num_001',
    '_style_corporate_num_002',
    '_style_encyclopedia_num_000',
    '_style_encyclopedia_num_001',
    '_style_news_num_000',
    '_style_news_num_001',
    '_style_news_num_002',
    '_style_news_num_003',
    '_style_news_num_004',
    '_style_social_num_000',
    '_style_social_num_001',
    '_style_social_num_002'
]
styles_main = ["blog", "corporate", "encyclopedia", "news", "social", "fictsheet"]
styles_nums = {"blog" : 2, "corporate" : 3, "encyclopedia" : 2, "news" : 5, "social" : 3, "fictsheet" : 1} # number of style variants

random.seed(10)
random.shuffle(styles_main)

event_ids = fqa_mc["event_id"].aggregate(pd.Series.unique)

questions_selected = {}
setups = {}
num_questions = {style : 0 for style in styles_main}
all_fictsheets = {}
all_fictions = {}

# for each event_id, one style: 
# training on the selected style, testing on selected questions per style

for i, event_id in enumerate(event_ids[:n_event_ids]):

    all_fictsheets[event_id] = load_fictsheets(event_id, fictsheets)

    # deterministically iterate over styles
    style_index = i%len(styles_main)
    style = styles_main[style_index]

    
    # style number such that maximum n_questions
    n_questions = {}
    for style_num in range(styles_nums[style]):
        fiction_id = f"{event_id}_style_{style}_num_00{style_num}"   
        n_questions[style_num] = len(fqa_mc[fqa_mc["fiction_id"] == fiction_id])

    max_style_num = max(n_questions, key=n_questions.get)
    fiction_id = f"{event_id}_style_{style}_num_00{max_style_num}"
    n_questions = len(fqa_mc[fqa_mc["fiction_id"] == fiction_id])


    # skip if no questions for any style number
    if n_questions == 0:
        print("No questions found for", fiction_id)
        continue

    num_questions[style] += n_questions

    fiction = fictions[fictions["fiction_id"] == fiction_id].iloc[0]["fiction"]
    all_fictions[fiction_id] = fiction
    questions = fqa_mc[fqa_mc["fiction_id"] == fiction_id]["question_id"].values
    questions_selected[fiction_id] = questions


    setups[event_id] = []

    for question_id in questions:
        mc_row = fqa_mc[fqa_mc["question_id"] == question_id].iloc[0].to_dict() 
        fict_row = fqa_row_to_data(data[data["question_id"] == question_id].iloc[0], data, fictsheets)

        setup = {
            "event_id": event_id,
            "fictsheet": fict_row["fictsheet"],
            "fiction_id": fiction_id,
            "fiction" : fict_row["fict"],
            "question_id": question_id,
            "question": list(fict_row["question"]),
            "topk_choices" : list(mc_row["topk_choices"]), # list len(.) = 4
            "target" : mc_row["target"],
        }

        setups[event_id].append(setup)

print("Number of questions in total per style: \t", num_questions)

Number of questions in total per style: 	 {'news': 4, 'encyclopedia': 4, 'corporate': 4, 'blog': 2, 'social': 1}


In [358]:
if not (n_event_ids == len(all_fictions.values()) and n_event_ids == len(all_fictsheets.values())):
    raise ValueError("Mismatch in number of event_ids, fictions and fictsheets")

# save setups
with open(f"setup/setups_fqa_{training_run_id}.json", "w") as f:
    json.dump(setups, f)

### Naive

In [374]:
fictsheets_memory_rag_naive = MemoryRAG()
fictions_memory_rag_naive = MemoryRAG()

In [375]:
for d in all_fictsheets.values():
    new_node = MemoryNodeRAG(input=d)
    fictsheets_memory_rag_naive.add(new_node)

print({len(node.embeddings) for node in fictsheets_memory_rag_naive.memory.values()}) # all 0
fictsheets_memory_rag_naive.get_nodes()
print({len(node.embeddings) for node in fictsheets_memory_rag_naive.memory.values()}) # all 1

for d in all_fictions.values():
    new_node = MemoryNodeRAG(input=d)
    fictions_memory_rag_naive.add(new_node)

print({len(node.embeddings) for node in fictions_memory_rag_naive.memory.values()}) # all 0
fictions_memory_rag_naive.get_nodes()
print({len(node.embeddings) for node in fictions_memory_rag_naive.memory.values()}) # all 1

{0}
{1}
{0}
{1}


In [376]:
fictsheets_memory_rag_naive.save(RAG_MEMORY_PATHS["fictsheets"])
fictions_memory_rag_naive.save(RAG_MEMORY_PATHS["fictions"])

In [ ]:

'''
# Verify

m = MemoryRAG()
m.load(RAG_MEMORY_PATHS["fictions"])

for node in m.memory.values():
    print(len(node.embeddings)) # all 1

m.recall("What was named after Olena Stepaniv in Lviv?", sensitivity=0.0, thres=0.0)
'''

### Student (TODO)

In [24]:
# Initialization

In [ ]:
teaching_prompt = "You are the world’s most studious detective of ficts, which are facts about fictitious stories that have never existed as facts about the real world. Memorize the facts explicitly. (NO verification required)"

student = StudentAgent(**AGENT_CONFIG)
student.reset_system_prompt(teaching_prompt, append=True)
student.save(TRAINING_STUDENT_MEMORY_PATH)

In [ ]:
# Teaching procedure

def train_memory(event, student, suffix):
    student.load(TRAINING_STUDENT_MEMORY_PATH+"_"+suffix)
    student.reset_chat()
    p = f"Ficts: {event}"
    student.run(p, remove_tools=["ask memory"])
    student.reset_chat()
    student.save(TRAINING_STUDENT_MEMORY_PATH+"_"+suffix)

In [ ]:
for j, context in enumerate(fictions.values()):
    print(j)
    train_memory(context, student, suffix="fictions")

In [ ]:
STUDENT_MEMORY_PATH = f"checkpoints/memory_{training_run_id}"

student.load(STUDENT_MEMORY_PATH)
student.reset_conversation()
student.save(STUDENT_MEMORY_PATH)

In [56]:
#student_wiki.memory_agent.memory.render()
#student_wiki.render_conversation()

# Training: Student vs Baselines

- StudentAgent:     all data --> memory --> ask

- pretraining ~     LLM
- answerable  ~     LLM + fact
- AgenticRAG  ~     LLM + RAG @ frozen memory
- NaiveRAG    ~     LLM + RAG @ data

In [380]:
old='''
def setup_experiments(data, n):
    experiments = []
    for i, row in data[:n].iterrows():
        
        event_id = ...
        fictsheet = 
        evals = json.loads(row["eval"]) if isinstance(row["eval"], str) else row["eval"]
        for question_type, qa in evals.items():
            question = qa["prompt"]
            correct_answer = qa["answer"]
            for agent_id in AGENT_IDS:
                if agent_id != "baseline_answerable":
                    continue
                
                exp = {
                    "run_id": run_id,
                    "agent_id": agent_id,
                    "event_id": event_id,
                    "fictsheet":
                    "question_id": question,
                    "target": correct_answer,
                    "topk_choices": [correct_answer],
                }
                experiments.append(exp)
'''